In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import statsmodels.api as sm

cwd = Path.cwd()
ROOT = cwd if (cwd / "requirements.txt").exists() else cwd.parent
if not (ROOT / "requirements.txt").exists():
    raise FileNotFoundError("Project root not found")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

PROCESSED = ROOT / "data" / "processed"
TABLES = ROOT / "reports" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)

from src.tr_vpin import build_volume_buckets, calculate_tr_vpin

with (ROOT / "research_config.json").open(encoding="utf-8") as file:
    config = json.load(file)

V = float(config["bucket_volume_btc"])
WINDOW = int(config["vpin_window"])
SPLIT = pd.Timestamp("2020-03-01", tz="UTC")

classified = pd.read_csv(PROCESSED / "btcusdt_1min_classified_2020_q1.csv")
for column in ["minute_start", "available_at"]:
    classified[column] = pd.to_datetime(classified[column], utc=True)
classified["tick_sign"] = pd.to_numeric(classified["tick_sign"], errors="raise")
classified["is_classified"] = (
    classified["volume_btc"].gt(0) & classified["tick_sign"].isin([-1, 1])
)

actual = pd.read_csv(PROCESSED / "btcusdt_tr_vpin_with_u2_2020_q1.csv")
actual["available_at"] = pd.to_datetime(actual["available_at"], utc=True)

train = pd.read_csv(PROCESSED / "forecast_train.csv")
test = pd.read_csv(PROCESSED / "forecast_test.csv")
for frame in (train, test):
    for column in ["timestamp", "target_end"]:
        frame[column] = pd.to_datetime(frame[column], utc=True)

# ---- placebo generator: permute signs among classified minutes within each UTC day
eligible_pos = np.flatnonzero(classified["is_classified"].to_numpy())
day = classified["minute_start"].dt.strftime("%Y-%m-%d").to_numpy()[eligible_pos]
day_groups = [eligible_pos[day == d] for d in np.unique(day)]


def placebo_classified(rng):
    signs = classified["tick_sign"].to_numpy(dtype=float).copy()
    for idx in day_groups:
        signs[idx] = rng.permutation(signs[idx])
    out = classified.copy()
    out["tick_sign"] = signs
    out["buy_volume_btc"] = np.where(signs == 1, out["volume_btc"], 0.0)
    out["sell_volume_btc"] = np.where(signs == -1, out["volume_btc"], 0.0)
    return out


check = calculate_tr_vpin(
    build_volume_buckets(placebo_classified(np.random.default_rng(0)), V), V, WINDOW
)
assert np.array_equal(check["bucket_id"].to_numpy(), actual["bucket_id"].to_numpy())
assert (
    check["available_at"].reset_index(drop=True)
    == actual["available_at"].reset_index(drop=True)
).all()
print("Placebo keeps bucket boundaries and availability times: OK")

# ---- forecasting helpers (same specification as the baseline)
TARGET = "rv_next_1h_pct"
CONTROLS = ["rv_prev_1h_pct", "log_volume_prev_1h"]
actual_target = test[TARGET].to_numpy(dtype=float)


def fit_predict(train_df, test_df, features):
    X_train = sm.add_constant(train_df[features].astype(float), has_constant="add")
    X_test = sm.add_constant(
        test_df[features].astype(float), has_constant="add"
    ).reindex(columns=X_train.columns)
    fit = sm.OLS(train_df[TARGET].astype(float), X_train, missing="raise").fit()
    return np.asarray(fit.predict(X_test))


def mse(forecast):
    return float(np.mean((forecast - actual_target) ** 2))


mse_controls = mse(fit_predict(train, test, CONTROLS))
gain_actual = 100 * (1 - mse(fit_predict(train, test, CONTROLS + ["tr_vpin"])) / mse_controls)
print("Actual TR-VPIN gain vs Controls, %:", round(gain_actual, 4))  # expect about 0.713

# ---- placebo repetitions (start with B = 5 to check run time)
B = 100
rng = np.random.default_rng(42)
u2_by_bucket = actual.set_index("bucket_id")["u2_vpin"]
rows = []

for rep in range(B):
    pv = calculate_tr_vpin(build_volume_buckets(placebo_classified(rng), V), V, WINDOW)
    pv["u2_vpin"] = pv["bucket_id"].map(u2_by_bucket)
    series = pv.set_index("bucket_id")["tr_vpin"]

    train_p = train.assign(tr_placebo=train["bucket_id"].map(series))
    test_p = test.assign(tr_placebo=test["bucket_id"].map(series))
    gain = 100 * (1 - mse(fit_predict(train_p, test_p, CONTROLS + ["tr_placebo"])) / mse_controls)

    row = {"rep": rep, "gain_pct": gain}
    for name, mask in [
        ("train", pv["available_at"].lt(SPLIT)),
        ("test", pv["available_at"].ge(SPLIT)),
    ]:
        part = pv.loc[mask]
        row[f"mean_tr_{name}"] = part["tr_vpin"].mean()
        row[f"corr_u2_{name}"] = part["tr_vpin"].corr(part["u2_vpin"])
    rows.append(row)

placebo = pd.DataFrame(rows)

actual_row = {"gain_pct": gain_actual}
for name, mask in [
    ("train", actual["available_at"].lt(SPLIT)),
    ("test", actual["available_at"].ge(SPLIT)),
]:
    part = actual.loc[mask].dropna(subset=["tr_vpin", "u2_vpin"])
    actual_row[f"mean_tr_{name}"] = part["tr_vpin"].mean()
    actual_row[f"corr_u2_{name}"] = part["tr_vpin"].corr(part["u2_vpin"])

metrics = [c for c in placebo.columns if c != "rep"]
summary = pd.DataFrame({
    "actual": pd.Series(actual_row),
    "placebo_mean": placebo[metrics].mean(),
    "placebo_p2_5": placebo[metrics].quantile(0.025),
    "placebo_p97_5": placebo[metrics].quantile(0.975),
}).loc[metrics]

print(summary.round(4).to_string())
print("Share of placebo gains >= actual gain:", float((placebo["gain_pct"] >= gain_actual).mean()))

summary.to_csv(TABLES / "placebo_vpin_summary.csv")
placebo.to_csv(TABLES / "placebo_vpin_repetitions.csv", index=False)

Placebo keeps bucket boundaries and availability times: OK
Actual TR-VPIN gain vs Controls, %: 0.7131
               actual  placebo_mean  placebo_p2_5  placebo_p97_5
gain_pct       0.7131        1.0309        0.4212         1.6980
mean_tr_train  0.2519        0.2372        0.2323         0.2431
corr_u2_train  0.9113        0.8618        0.8121         0.8975
mean_tr_test   0.3415        0.3419        0.3349         0.3482
corr_u2_test   0.9352        0.9439        0.9261         0.9588
Share of placebo gains >= actual gain: 0.8


## Interpretation

Signs of classified minutes were permuted within each UTC day (B = 100). Volumes,
bucket boundaries and availability times are unchanged.

- In March the mean TR-VPIN (0.3415) and its correlation with U2-VPIN (0.9352) lie
  inside the placebo ranges, so tick-rule signs add nothing detectable beyond
  trading-volume structure in the test period.
- In January–February the mean (0.2519) and the correlation (0.9113) are above the
  placebo ranges, so there is a small amount of sign persistence in the calm period.
- The 0.71% MSE gain of Controls + TR-VPIN is inside the placebo range
  (0.42%–1.70%; the placebo mean is 1.03%; 80% of placebo gains are at least as large).